In [ ]:
# So we can run this from python2 or python3.
# But you should be using python3!!!
from __future__ import print_function

# Submit

Tools run on the hubs are limited because they run in a container on an execution host.  Many simulations require large amounts or memory and/or CPU time.  They require access to external HPC or grid resources.  

**submit** takes a user command and executes it remotely. The objective is to allow the user to issue a command in the same manner as a locally executed command. Multiple submission mechanisms are available for run dissemination. A set of steps are executed for each run submission:

* Destination site is selected
* A wrapper script is generated for remote execution.

* If needed a batch system description file is generated.
* Input files for a run are gathered and transferred to the remote site. Transferred files include the wrapper and batch description scripts.
* The wrapper script is executed remotely.
* Progress of the remote run is monitored until completion.
* Output files from the run are returned to the dissemination point.


```bash
> submit --help
Usage: submit [options]

Options:
  -h, --help            Report command usage. Optionally request listing of
                        managers, tools, venues, or examples.
  -l, --local           Execute command locally
  --status              Report status for runs executing remotely.
  -k, --kill            Kill runs executing remotely.
  -v, --venue           Remote job destination
  -i, --inputfile       Input file
  -p, --parameters      Parameter sweep variables. See examples.
  -d, --data            Parametric variable data - csv format
  -s SEPARATOR, --separator=SEPARATOR
                        Parameter sweep variable list separator
  -n NCPUS, --nCpus=NCPUS
                        Number of processors for MPI execution
  -N PPN, --ppn=PPN     Number of processors/node for MPI execution
  -w WALLTIME, --wallTime=WALLTIME
                        Estimated walltime hh:mm:ss or minutes
  -e, --env             Variable=value
  --runName=RUNNAME     Name used for directories and files created during the
                        run. Restricted to alphanumeric characters
  -m, --manager         Multiprocessor job manager
  -r NREDUNDANT, --redundancy=NREDUNDANT
                        Number of indentical simulations to execute in
                        parallel
  -M, --metrics         Report resource usage on exit
  -Q, --quota           Enforce local user quota on remote execution host
  -q, --noquota         Do not enforce local user quota on remote execution
                        host
  --tailStdout          Periodically report tail of stdout file.
  --tailStderr          Periodically report tail of stderr file.
  --tail                Periodically report tail of application file.
  --progress            Show progress method. Choices are auto, curses,
                        submit, text, pegasus, or silent.
```


# Learning by Doing

This notebook will demonstrate how to use submit.  First it will show command line usage that will not require Python.  Then it will show some how to use a python convenience function to run submit.


## Venues

Submit can submit to different hosts or clusters. You can get a list for your hub by asking submit:


In [ ]:
!submit --help venues

<div class="alert alert-warning">
How do we know what the default venue is?
How should we choose the venue to use?
</div>

### What commands can I submit?

#### Submitting locally with --local

Any executable can be used when submitting locally.  Local submit runs in the current
execution host in the current container, so there are few reasons to use it.  However, it
does run quickly, so it is useful for quick tests such as the ones we run in this notebook.

#### Submitting to the grid

Only executables staged in /apps
can be submitted to the grid.  

In [ ]:
!submit --local echo hi

## Submitting with Parameters

Parameter examples:

submit -p @@cap=10pf,100pf,1uf sim.exe @:indeck

    Submit 3 jobs. The @:indeck means "use the file indeck as a template
    file." Substitute the values 10pf, 100pf, and 1uf in place of @@cap within the
    file. Send off one job for each of the values and bring back the results.

submit -p @@vth=0:0.2:5 -p @@cap=10pf,100pf,1uf sim.exe @:indeck

    Submit 78 jobs. The parameter @@vth goes from 0 to 5 in steps of 0.2,
    so there are 26 values for @@vth. For each of those values, the parameter
    @@cap changes from 10pf to 100pf to 1uf. 26 x 3 = 78 jobs total. Again
    @:indeck is treated as a template, and the values are substituted in place of
    @@vth and @@cap in that file.

submit -p params sim.exe @:indeck

    In this case, parameter definitions are taken from the file named
    params instead of the command line. The file might have the following
    contents:

        # paramters for my job submission
        parameter @@vth=0:0.2:5
        parameter @@cap = 10pf,100pf,1uf

submit -p "params;@@num=1-10;@@color=blue" job.sh @:job.data

    For someone who loves syntax and complexity... The semicolon separates
    the parameters value into three parts. The first says to load parameters from
    a file params. The next part says add an additional parameter @@num that goes
    from 1 to 10. The last part says add an additional parameter @@color with a
    single value blue. The parameters @@num and @@color cannot override anything
    defined within params; they must be new parameter names.

submit -d input.csv sim.exe @:indeck

    Takes parameters from the data file input.csv, which must be in comma-
    separated value format. The first line of this file may contain a series of
    @@param names for each of the columns. Whitespace is significant for all
    values entered in the csv file. If it doesn't, then the columns are assumed to
    be called @@1, @@2, @@3, etc. Each of the remaining lines represents a set of
    parameter values for one job; if there are 100 such lines, there will be 100
    jobs. For example, the file input.csv might look like this:

        @@vth,@@cap
        1.1,1pf
        2.2,1pf
        1.1,10pf
        2.2,10pf

    Parameters are substituted as before into template files such as
    @:indeck.

submit -d input.csv -p "@@doping=1e15-1e17 in 30 log" sim.exe @:infile

    Takes parameters from the data file input.csv, but also adds another
    parameter @@doping which goes from 1e15 to 1e17 in 30 points on a log scale.
    For each of these points, all values in the data file will be executed. If the
    data file specifies 50 jobs, then this command would run 30 x 50 = 1500 jobs.

submit -d input.csv -i @:extra/data.txt sim.exe @:indeck

    In addition to the template indeck file, send along another file
    extra/data.txt with each job, and treat it as a template too.

submit -s / -p @@address=23 Main St.,Hometown,Indiana/42 Broadway,Hometown,Indiana -s , -p @@color=red,green,blue job.sh @:job.data

    Change the separator to slash when defining the addresses, then change
    back to comma for the @@color parameter and any remaining arguments. We
    shouldn't have to change the separator often, but it might come in handy if
    the value strings themselves have commas.

submit -p @@num=1:1000 sim.exe input@@num

    Submit jobs 1,2,3,...,1000. Parameter names such as @@num are
    recognized not only in template files, but also for arguments on the command
    line. In this case, the numbers 1,2,3,...,1000 are substituted into the file
    name, so the various jobs take their input from "input1", "input2", ...,
    "input1000".

submit -p @@file=glob:indeck* sim.exe @@file

    Look for files matching indeck* and use the list of names as the
    parameter @@file. Those values could be substituted into other template files,
    or used on the command line as in this example. Suppose the directory contains
    files indeck1, indeck10, and indeck2.  The glob option will order the files in
    a natural order: indeck1, indeck2, indeck10.  This example would launch three
    jobs using each of those files as input for the job.

submit -p @@file=globnat:indeck* sim.exe @@file

    This option has been deprecated.  The functionality is now available with the glob option.

Let's try the echo program with a list of different input parameters.  To see progress, we need to use "text" or "submit".  Other options won't work well with Jupyter.


In [ ]:
!submit --local --progress text -p @@name=hub1,hub2,hub3 echo @@name

Notice how the outputs were written to subdirectories in a directory that was created for us.
If we want to access those results more conveniently, it would be best to tell submit what name to use for the directory.
We can do this by using the **--runName** parameter.
<div class="alert alert-info">
If the directory exists, submit will complain and exit immediately.

</div>

In [ ]:
!rm -rf echotest
!submit --local --runName=echotest --progress submit -s, -p @@name=hub1,hub2,hub3 echo @@name

Once submit finishes, we can see the standard output for each job in the "echotext" subdirectory.  Standard output  will be in echotest_{jobnum}.stdout.

In [ ]:
!ls -lR echotest

In [ ]:
for i in range(1,4):
    with open("echotest/%02d/echotest_%02d.stdout" % (i, i), 'r') as f:
        print("Run %02d: %s" % (i, f.read()))

In [ ]:
!cat echotest/parameterCombinations.csv

# Using runCommand

Using the exclamation point(!) to run shell commands from Jupyter is fine for many cases.  But often you want to call submit from a python function or check the return value.  hublib provides some very flexible python functions to do this  https://hubzero.github.io/hublib/cmd.html

In [ ]:
from hublib.cmd import runCommand
!rm -rf runtest
res, stdout, stderr = runCommand("submit --runName=runtest --progress submit -p @@Vin=1,2,3,4,5 /apps/pegtut/current/examples/capacitor_voltage/sim1.py  --Vin @@Vin")

Return code for normal completion is zero.

In [ ]:
res

For the last example, we used a test program installed on nanohub so we didn't have to execute locally.  The program, "sim1.py" takes a single argument and outputs values to a file, "out.log".  It does not write to stdout.  You can see all the output in the subdirectories under "runtest"

In [ ]:
!ls -lR runtest

# Using SubmitCommand class

Furthur Python integration is achieved by using the hubzero.submit.SubmitCommand class.  Methods are provided for specifying any and all **submit** arguments.  For each **submit** command argument there are typically two method types - set and reset. The standard python **help** builtin can be used to show methods and associated arguments.  The code shown here can also be used by Rappture Python wrapper scripts.

In [ ]:
import os
from hubzero.submit.SubmitCommand import SubmitCommand

submitCommand = SubmitCommand()
help (submitCommand)

Help about the underlying **submit** command can be easily reported.

In [ ]:
submitCommand = SubmitCommand()
result = submitCommand.submit(['--help'])

A listing of available venues can be listed

In [ ]:
submitCommand = SubmitCommand()
submitCommand.setHelp(detail='venues')
result = submitCommand.submit()

A listing of available staged tools can be listed

In [ ]:
submitCommand = SubmitCommand()
submitCommand.setHelp(detail='tools')
result = submitCommand.submit()

For demonstration purposes we can use a Python application that is part of the Pegasus Tutorial tool. Because it is installed in the /apps directory it is available for submisison to most venues.

In [ ]:
applicationCode = '/apps/pegtut/current/examples/capacitor_voltage/sim1.py'

The SubmitCommand class has a method that excepts command arguments as a simple list.

In [ ]:
submitCommand = SubmitCommand()
result = submitCommand.submit(['-r','3','-w','5',applicationCode,'--C','0.001','--Vin','3'])
print(result)

You should notice that the single run resulted in submission of jobs to three different sites.  When no venue is specified and the application is not staged at any remote venue, redundant jobs may be submited.  When one job completes sucessfully the other jobs are terminated.
The result dictionary has three members.  The *jobId* is the unique identifier given by **submit** to every run.  *runName* is the name of the run and by default it matches the *jobId*.  *runName* can be optionally set with **submit** arguments.  Standard output and standard error files generated by remote application execution are returned as *runName*.stdout and *runName*.stderr.  The standard error file will not be returned if it is empty. 

In [ ]:
runStdout = result['runName'] + '.stdout'
!cat $runStdout

In [ ]:
runStderr = result['runName'] + '.stderr'
!cat $runStderr

Using the set methods and specifying a particular venue

In [ ]:
submitCommand = SubmitCommand()
submitCommand.setWallTime(5)
submitCommand.setVenue('OSG')
submitCommand.setCommand(applicationCode)
submitCommand.setCommandArguments(['--C','0.001','--Vin','3'])
submitCommand.setStdin(os.devnull)
submitCommand.show()
result = submitCommand.submit()

Here is an example with a parameter sweep over two variables.  Parameter sweep results are returned in a directory named *runName*.  Each job in the sweep has a separate subdirectory.

In [ ]:
submitCommand = SubmitCommand()
submitCommand.setWallTime(5)
submitCommand.setParameters(['@@Vin=1:2:10','@@C=100e-6,100e-5'])
submitCommand.setCommand(applicationCode)
submitCommand.setCommandArguments(['--C','@@C','--Vin','@@Vin'])
submitCommand.setStdin(os.devnull)
submitCommand.show()
result = submitCommand.submit()

runResultsDir = result['runName']
!ls -R $runResultsDir

Alternative progress reporting methods are provided to facilitate advanced reporting. As an example the *submit* progress reporting method output is injested by Rappture and rendered as a progress bar.

In [ ]:
submitCommand = SubmitCommand()
submitCommand.setWallTime(5)
submitCommand.setParameters(['@@Vin=1:2:10','@@C=100e-6,100e-5'])
submitCommand.setProgress(detail='submit')
submitCommand.setCommand(applicationCode)
submitCommand.setCommandArguments(['--C','@@C','--Vin','@@Vin'])
submitCommand.setStdin(os.devnull)
submitCommand.show()
result = submitCommand.submit()

Parameter sweep variable values can also be read from a file.

In [ ]:
with open("input.csv",'w') as fpInput:
   fpInput.write("@@Vin,@@C\n")
   for v in range(1,6):
      for c in (0.0001,0.001):
         fpInput.write("%f,%f\n" % (v,c))
!cat input.csv

In [ ]:
submitCommand = SubmitCommand()
submitCommand.setWallTime(5)
submitCommand.setDataFile('input.csv')
submitCommand.setCommand(applicationCode)
submitCommand.setCommandArguments(['--C','@@C','--Vin','@@Vin'])
submitCommand.setStdin(os.devnull)
submitCommand.show()
result = submitCommand.submit()

The mapping of variable combinations to job number is contained in the parameterCombinations.csv file.

In [ ]:
combinationsFile = os.path.join(result['runName'],'parameterCombinations.csv')
!cat $combinationsFile